# OBJ to BREP — Narkomfin Building (Type K)

In [ ]:
from pathlib import Path
from topologicpy.Topology import Topology
from topologicpy.Cluster  import Cluster

HERE = Path.cwd()  # VS Code sets CWD to the notebook's folder

OBJ_PATH_K  = HERE.parent / '02_graph_analysis' / 'assets' / 'TheNarkomfinHouse-01.obj'
BREP_PATH_K = HERE.parent / '02_graph_analysis' / 'output'  / 'L1_narkomfin_type_k.brep'

topo = Topology.ByOBJPath(str(OBJ_PATH_K))
if isinstance(topo, list):
    topo = Cluster.ByTopologies(topo)
print(f'Type K — Type: {Topology.TypeAsString(topo)}  Faces: {len(Topology.Faces(topo))}')
ok = Topology.ExportToBREP(topo, path=str(BREP_PATH_K), overwrite=True)
print(f'Type K — Exported: {ok}')

Type K — Type: Cluster  Faces: 19
Type K — Exported: True


In [ ]:
from pathlib import Path
from topologicpy.Topology import Topology
from topologicpy.Cluster  import Cluster
from topologicpy.Shell    import Shell
from topologicpy.Face     import Face
from topologicpy.Wire     import Wire
from topologicpy.Dictionary import Dictionary

HERE = Path.cwd()

BREP_IN_K  = HERE.parent / '02_graph_analysis' / 'output' / 'L1_narkomfin_type_k.brep'
BREP_OUT_K = HERE.parent / '02_graph_analysis' / 'output' / 'L1_narkomfin_type_k_face.brep'

def brep_to_face(brep_in, brep_out, label):
    floor_plan = Topology.ByBREPPath(str(brep_in))
    triangles  = Cluster.Faces(floor_plan)
    shell      = Shell.ByFaces(triangles)
    eb         = Shell.ExternalBoundary(shell)
    ib_list    = Shell.InternalBoundaries(shell)

    # Handle disconnected components: if there are multiple external
    # boundary wires, keep only the longest one (the main building outline)
    eb_wires = Topology.Wires(eb)
    if len(eb_wires) > 1:
        print(f'  {label}: {len(eb_wires)} disconnected boundaries — keeping longest.')
        eb      = max(eb_wires, key=lambda w: Wire.Length(w))
        ib_list = []  # holes from the discarded piece are no longer valid

    face = Face.ByWires(eb, ib_list)
    face = Topology.RemoveCollinearEdges(face)
    print(f'{label} — edges: {len(Topology.Edges(face))}')
    Topology.ExportToBREP(face, path=str(brep_out), overwrite=True)
    return face

face_k = brep_to_face(BREP_IN_K, BREP_OUT_K, 'Type K')
Topology.Show(face_k)

Shell.ByFaces - Error: Could not create shell. Returning None.
Shell.ExternalBoundary - Error: The input shell parameter is not a valid Shell. Returning None.
Shell.InternalBoundaries - Error: The input shell parameter is not a valid Shell. Returning None.
Topology.Wires - Error: The input is not a valid topology. Returning None


TypeError: object of type 'NoneType' has no len()